# Lab 05 — Ensembles and the ablation study

*Machine Learning and Data Mining — Claudio Sartori, University of Bologna*

## Goal of this lab

Lecture 13 ended with a table: on NordicMart, bagging, random forests and boosting all reach
the logistic regression of Lecture 11, none goes clearly beyond it. Today we rebuild those models, and we ask a question
that a single score cannot answer: **which part of a model actually does the work?**
The tool is the **ablation study**: remove one component, retrain, measure again, compare.

| # | Step | Where |
|---|---|---|
| 1 | What an ablation study is, and a helper function to run one | A |
| 2 | Ablation of the **preparation** of Lab 4 (logistic regression) | A |
| 3 | Bagging and random forests; ablation of the **two sources of randomness**; number of trees | B |
| 4 | Gradient boosting: number of trees and **depth**, compared with the forest | C |
| 5 | Ablation of the **attributes** (drop-column), single and in groups | D |
| 6 | One look at the test set | E |
| ★ | AdaBoost's probabilities; boosting with missing values; drop-column vs permutation importance | F |

- **Part A (≈ 25 min)** — 15 minutes of introduction, then the first ablation.
- **Part B (≈ 30 min)** — bagging and random forests.
- **Part C (≈ 20 min)** — gradient boosting.
- **Part D (≈ 20 min)** — which attributes matter.
- **Part E (≈ 5 min)** — the test set, once.
- About 10 minutes are left for the ★ exercises or for a longer discussion of the results.
- Exercises marked ★ are optional: do them if you have time, or at home.

### How to work (as in Labs 1–4)

- Write code in the empty cells; write **answers in markdown cells**, not in comments.
- Raw data are never modified: every change goes into a copy with a new name.
- Every random step gets a fixed `random_state`, so that your numbers are the same as the slides'.
- Every choice and every ablation is made with **cross-validation on the training part**. The test set is opened once, in Part E.
- Some cells take 10–30 seconds: ensembles train hundreds of trees, and an ablation retrains the model once per variant and per fold.
- Before closing: **Kernel → Restart & Run All**.

## Setup

New imports: the ensembles of Lecture 13 (`sklearn.ensemble`) and the permutation importance (`sklearn.inspection`).
The data, the split, the folds, the `prepare` function and the cost function are those of Lab 4: run the cells, do not change them.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix
import sklearn

DATA_DIR = Path("../../data")
NORDICMART_FILE = DATA_DIR / "nordicmart_customers.csv"
SEED = 42

pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (8, 4)

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

scikit-learn 1.9.1 | pandas 3.0.5


In [2]:
df = pd.read_csv(NORDICMART_FILE)
FEATURES = ["tier", "age", "days_since_last_buy", "annual_spend", "web_sessions_month", "tickets_90d"]
Xn = df[FEATURES].copy()
Xn["tier"] = Xn["tier"].map({"Basic": 0, "Silver": 1, "Gold": 2})
yn = df["churned"]

Xn_train, Xn_test, yn_train, yn_test = train_test_split(Xn, yn, test_size=0.3, stratify=yn, random_state=SEED)
cv_n = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def prepare(X):                                    # Lab 4, C1: fixed rules from the EDA log, nothing learned from data
    return pd.DataFrame({
        "tier": X["tier"],
        "age": X["age"].replace(0, np.nan),
        "recency": X["days_since_last_buy"].replace(999, np.nan),
        "never_bought": (X["days_since_last_buy"] == 999).astype(int),
        "log_spend": np.log10(X["annual_spend"]),
        "sessions": X["web_sessions_month"].fillna(0),
        "store_only": X["web_sessions_month"].isna().astype(int),
        "tickets": X["tickets_90d"]}, index=X.index)

Xp_train, Xp_test = prepare(Xn_train), prepare(Xn_test)

def pipe(model):                                   # Lab 4, C2: learned preparation inside the pipeline
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), model)

def churn_cost(y_true, y_pred):                    # Lab 4, D1: EUR 10 per false alarm, EUR 100 per missed churner
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return 10 * fp + 100 * fn

print(f"training: {len(yn_train)} customers, {yn_train.sum()} churners | test: {len(yn_test)} customers, {yn_test.sum()} churners")

training: 2100 customers, 252 churners | test: 900 customers, 108 churners


As in Lecture 13, **every** model receives the prepared columns through the same pipeline, trees included: imputation
is needed by most models, and standardization does not change a tree (it only moves its thresholds). So every model sees
exactly the same data, and differences come from the models.

# Part A — The ablation study

## A0. What an ablation study is

**The word.** In neuroscience, an *ablation* is the removal of a part of the brain, to see which behaviour is lost. In machine
learning the term was borrowed for an experiment on a *system* made of several parts: **remove one component, retrain,
measure again**. The drop in performance is the contribution of that component — *in the presence of all the others*.

**What can be a component.**

| kind | examples on NordicMart |
|---|---|
| input attributes, single or in groups | `recency`; the pair `sessions` + `store_only` |
| preparation steps | the log of the spend, the indicator columns, standardization |
| parts of the algorithm | bootstrap and random attribute subsets in a forest; shrinkage in boosting |
| regularizers | the minimum leaf size of a tree; the learning rate of boosting |
| size of the ensemble | number of trees |

**The protocol.**

1. A **baseline**: the full system, as it would be delivered.
1. One **variant** per component, with that component removed (or switched off) and **everything else identical**:
   same data, same folds, same measure, same `random_state`.
1. Every variant is **retrained** from scratch and evaluated with cross-validation on the training part
   (the test set stays closed: an ablation is part of building the model, Lecture 10a).
1. Report the **difference** from the baseline, fold by fold, and how consistent it is.

**Why fold by fold.** The five CV scores of a model differ from each other for two reasons:

- **the fold**: each fold has different test records, and some folds happen to be harder than others *for every model*
  (e.g. a fold with many churners that behave like loyal customers);
- **the model**: what we actually want to measure.

The first reason is **shared**: if fold 3 is hard, it is hard for both variants, and both scores go down together. So the standard deviation
of the five scores of one model mostly measures how different the folds are, not how good the model is. Comparing two models through
"mean ± std" mixes the two reasons, and two clearly different models can look indistinguishable.

The remedy is to compare the two scores **of the same fold**, and only then average. A small invented example with three folds:

| | fold 1 | fold 2 | fold 3 | mean ± std |
|---|---|---|---|---|
| variant A | 0.80 | 0.70 | 0.90 | 0.80 ± 0.08 |
| variant B | 0.82 | 0.73 | 0.91 | 0.82 ± 0.07 |
| **B − A** | +0.02 | +0.03 | +0.01 | **+0.02 ± 0.01** |

The two intervals overlap widely (0.72–0.88 and 0.75–0.89): taken separately, A and B seem equivalent. But fold by fold, B beats A
**every time**, by 0.01–0.03: the large ups and downs (0.70 vs 0.90) belong to the folds and cancel in the difference. The noise of the
difference (± 0.01) is much smaller than the noise of each score (± 0.08) because the two scores **move together**. In formulas:
$\mathrm{Var}(B - A) = \mathrm{Var}(A) + \mathrm{Var}(B) - 2\,\mathrm{Cov}(A, B)$, and the covariance is large when the folds dominate.

This is called a **paired** comparison: each fold gives a *pair* of scores measured on the same test records, as when a treatment is
judged by measuring the *same* patients before and after rather than two different groups. It works only if every variant uses
**exactly the same folds**: this is why `cv_n` is created once, with a fixed `random_state`, and passed to every call.
In A1 you will check all this on the real numbers of NordicMart.

**Reading the result.**

- large drop, the same sign on every fold → the component matters;
- difference ≈ 0 → the component can be removed: **a simpler system, same performance**;
- a *rise* → the component hurts: remove it.

**Three pitfalls** (we meet all three today).

- **Redundancy**: two components carry the same information; removing either one shows nothing, removing both shows a lot.
  Ablate **groups** too.
- **Interactions**: the effect of removing A depends on whether B is there. Effects are not additive: a 2 × 2 design
  (with/without A × with/without B) shows it.
- **Scope**: an ablation measures what *this* model, trained in *this* way, gets from a component. It is not a statement about the
  world ("recency causes churn"), and another model may use the same attribute differently.

**Ablation vs the importances of Lecture 13.** Permutation importance shuffles a column and *does not retrain*: it measures how
much a *fitted* model relies on that column. A drop-column ablation *retrains without the column*: it measures how much the
*learning* needs it — if another column can replace it, the drop is small. Both are useful; they answer different questions.

## A1. Paired folds, and a helper function

1. With `cross_val_score(..., cv=cv_n, scoring="roc_auc")` compute the five fold AUCs of two models on `Xp_train`:
   the tree of Lecture 13, `pipe(DecisionTreeClassifier(max_depth=5, min_samples_leaf=30, random_state=0))`, and the logistic regression
   of Lecture 13, `pipe(LogisticRegression(C=0.1, max_iter=2000))`. Print the five scores of each, their mean and standard deviation,
   and the same for the five **differences** (LR − tree).
1. Write a function `ablation(variants)`:
   - `variants` is a dictionary `{name: (estimator, X)}`; the **first** entry is the baseline;
   - for every variant compute the five fold AUCs on `(X, yn_train)` with `cv_n`;
   - return a DataFrame with one row per variant and columns `CV AUC` (mean), `std`, `diff` (mean of the fold-by-fold difference
     from the baseline), `std diff` (its standard deviation) and `folds worse` (in how many folds the variant is below the baseline).

We will use `ablation` in every part of the lab.

In [3]:
tree_13 = pipe(DecisionTreeClassifier(max_depth=5, min_samples_leaf=30, random_state=0))
lr_13 = pipe(LogisticRegression(C=0.1, max_iter=2000))

s_tree = cross_val_score(tree_13, Xp_train, yn_train, cv=cv_n, scoring="roc_auc")
s_lr = cross_val_score(lr_13, Xp_train, yn_train, cv=cv_n, scoring="roc_auc")
folds = pd.DataFrame({"tree": s_tree, "logistic regression": s_lr, "difference": s_lr - s_tree},
                     index=[f"fold {k}" for k in range(1, 6)]).T
folds["mean"], folds["std"] = folds.mean(axis=1), folds.std(axis=1, ddof=0)
display(folds)

def ablation(variants):
    scores = {name: cross_val_score(est, X, yn_train, cv=cv_n, scoring="roc_auc") for name, (est, X) in variants.items()}
    base = next(iter(scores.values()))
    return pd.DataFrame({name: {"CV AUC": s.mean(), "std": s.std(), "diff": (s - base).mean(),
                                "std diff": (s - base).std(), "folds worse": int((s < base).sum())}
                         for name, s in scores.items()}).T.astype({"folds worse": int}).round(3)

ablation({"logistic regression": (lr_13, Xp_train), "tree": (tree_13, Xp_train)})

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
tree,0.779,0.803,0.760,0.694,0.793,0.766,0.039
logistic regression,0.818,0.869,0.827,0.762,0.841,0.823,0.035
difference,0.038,0.066,0.068,0.068,0.048,0.058,0.012


,CV AUC,std,diff,std diff,folds worse
logistic regression,0.823,0.035,0.000,0.000,0
tree,0.766,0.039,-0.058,0.012,5


**Questions.**

1. Which fold is the hardest for both models? What happens to it in the row of the differences?
1. The two CV AUCs are 0.823 ± 0.035 and 0.766 ± 0.039: the two intervals mean ± std overlap. Is the logistic regression better or not?

**Answers.**

1. Fold 4: 0.694 for the tree, 0.762 for logistic regression — both models find it hard (that fold happens to contain churners that are
   difficult to rank). In the differences fold 4 is an ordinary 0.068, close to folds 2 and 3: the difficulty of the fold cancels out.
1. Better, clearly. The standard deviations of the two scores (≈ 0.035) mostly measure how different the folds are, and that is the same for
   both models. The difference is +0.058 with a standard deviation of only 0.012, and it is positive in all five folds. This is why an
   ablation compares variants **on the same folds**, and looks at `diff` and `std diff`, not at two separate `std`.

## A2. Ablation of the preparation

In Lab 4 we prepared the attributes with fixed rules (`prepare`) and then imputed and standardized inside the pipeline. Was every step
needed? Take the logistic regression `lr_13` as the model and build these variants, each removing **one** step (or two, where stated):

| variant | estimator | data |
|---|---|---|
| full preparation (baseline) | `lr_13` | `Xp_train` |
| no standardization | `make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(C=0.1, max_iter=5000))` | `Xp_train` |
| spend not in log | `lr_13` | `Xp_train` with `log_spend` replaced by the raw `annual_spend` |
| no indicator columns | `lr_13` | `Xp_train` without `never_bought` and `store_only` |
| 999 kept as a recency | `lr_13` | `Xp_train` with `recency` replaced by the raw `days_since_last_buy` |
| 999 kept, no `never_bought` | `lr_13` | as above, and without `never_bought` |
| no preparation at all | `lr_13` | the raw columns `Xn_train` |

Raw values come from `Xn_train` (same index as `Xp_train`). Use `.assign(...)` or `.drop(columns=...)` on a copy: never modify `Xp_train`.

In [4]:
prep_variants = {
    "full preparation": (lr_13, Xp_train),
    "no standardization": (make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(C=0.1, max_iter=5000)), Xp_train),
    "spend not in log": (lr_13, Xp_train.assign(log_spend=Xn_train["annual_spend"])),
    "no indicator columns": (lr_13, Xp_train.drop(columns=["never_bought", "store_only"])),
    "999 kept as a recency": (lr_13, Xp_train.assign(recency=Xn_train["days_since_last_buy"])),
    "999 kept, no never_bought": (lr_13, Xp_train.assign(recency=Xn_train["days_since_last_buy"]).drop(columns="never_bought")),
    "no preparation at all": (lr_13, Xn_train)}
ablation(prep_variants)

,CV AUC,std,diff,std diff,folds worse
full preparation,0.823,0.035,0.000,0.000,0
no standardization,0.823,0.037,-0.001,0.003,3
spend not in log,0.822,0.035,-0.002,0.001,4
no indicator columns,0.817,0.037,-0.007,0.006,4
999 kept as a recency,0.818,0.035,-0.005,0.006,4
"999 kept, no never_bought",0.757,0.029,-0.066,0.011,5
no preparation at all,0.720,0.044,-0.103,0.025,5


**Questions.**

1. Which single steps could be removed without loss? Which one would you keep anyway, and why?
1. "999 kept as a recency" costs 0.005, removing `never_bought` alone costs almost nothing (Part D will confirm it), but doing both costs 0.066.
   Explain the interaction: what does the logistic regression do with a customer whose recency is 999 when the flag is there, and when it is not?
1. "No preparation at all" loses about 0.10 of AUC. Is it the sum of the single losses? What does this tell you about ablating one step at a time?

**Answers.**

1. Standardization (−0.001), the log of the spend (−0.002) and, for this model, even the indicator columns (−0.007, though worse in 4 folds of 5) change
   almost nothing. Standardization is kept anyway: it costs nothing, it makes the penalty `C` treat all attributes alike (Lecture 11),
   and kNN and SVM need it. A component that does not help *this* model is not automatically useless for the whole pipeline.
1. With the flag, the 999 of the never-buyers (1.8% of the customers) is still an extreme value of `recency`, but the model has a separate
   weight for `never_bought` that can correct the prediction for exactly those customers. Without the flag, the only way to fit them is through
   the recency weight, and a value 999 among recencies of a few dozen days drags the whole line: the weight of the most important attribute is
   distorted for **everybody**. Each step alone is harmless because the other one compensates: an **interaction**.
1. No: the single losses sum to about 0.02, the total is 0.10 (on fold 1 even 0.14). Steps compensate each other — `prepare` contains two such
   pairs (999 + flag, missing sessions + flag) — so removing one at a time underestimates their value. When components can stand in for each
   other, ablate them **together** as well.

# Part B — Bagging and random forests

## B1. Three models of Lecture 13

Compute with `ablation` (baseline: the forest) the CV AUC of the three tree models of Lecture 13, *NordicMart: six models*, all with `pipe(...)` on `Xp_train`:

| name | model |
|---|---|
| random forest | `RandomForestClassifier(n_estimators=300, max_features=1, min_samples_leaf=5, random_state=0, n_jobs=-1)` |
| bagging | `BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=50), n_estimators=300, random_state=0, n_jobs=-1)` |
| single tree | `tree_13` |

These are the winners of the grid searches of the lecture: we do not repeat the searches (they take minutes). `n_jobs=-1` uses all the cores.

Then fit the forest on the whole training part and compute its AUC **on the training part** itself.

In [5]:
rf_13 = pipe(RandomForestClassifier(n_estimators=300, max_features=1, min_samples_leaf=5, random_state=0, n_jobs=-1))
bag_13 = pipe(BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=50), n_estimators=300, random_state=0, n_jobs=-1))
display(ablation({"random forest": (rf_13, Xp_train), "bagging": (bag_13, Xp_train), "single tree": (tree_13, Xp_train)}))

rf_13.fit(Xp_train, yn_train)
print(f"random forest, AUC on its own training data: {roc_auc_score(yn_train, rf_13.predict_proba(Xp_train)[:, 1]):.3f}")

,CV AUC,std,diff,std diff,folds worse
random forest,0.812,0.028,0.000,0.000,0
bagging,0.801,0.026,-0.011,0.005,5
single tree,0.766,0.039,-0.046,0.018,5


random forest, AUC on its own training data: 0.956


**Question.** Training AUC 0.96, CV AUC 0.81: is the forest overfitting? Should you reduce the number of trees? (Lecture 13, exit question.)

**Answer.** The gap only says that the forest has seen the training customers: each of its trees fits its bootstrap sample closely (leaves of 5).
What matters is the CV AUC, which is the best of the three tree models. Reducing the number of trees would not help: in a forest more trees
never increase overfitting (B4). To act on overfitting you change the **size of the trees** (`min_samples_leaf`, `max_depth`) or `max_features`.

## B2. Out-of-bag estimate

Each tree of the forest is trained on a bootstrap sample: about 37% of the training customers are *out of bag* for it.
With `oob_score=True` the forest collects, for every training customer, the vote of the trees that did not see it.

1. Fit `pipe(RandomForestClassifier(..., oob_score=True))` (same parameters as `rf_13`) on the training part.
   The forest is the last step of the pipeline: `model[-1]`.
1. `oob_decision_function_[:, 1]` contains the out-of-bag probability of churn of each training customer: compute the **OOB AUC**.
1. Compare it with the CV AUC of B1. Which one needed more training?

In [6]:
rf_oob = pipe(RandomForestClassifier(n_estimators=300, max_features=1, min_samples_leaf=5, oob_score=True,
                                     random_state=0, n_jobs=-1)).fit(Xp_train, yn_train)
p_oob = rf_oob[-1].oob_decision_function_[:, 1]
print(f"OOB AUC {roc_auc_score(yn_train, p_oob):.3f} | OOB accuracy {rf_oob[-1].oob_score_:.3f}")

OOB AUC 0.798 | OOB accuracy 0.893


**Answer.** OOB AUC 0.798, CV AUC 0.812: the same order, a little lower. The OOB estimate cost **one** fit of the forest; CV needed five. It is a bit
pessimistic because each customer is judged by about 110 trees instead of 300, and with fewer trees the forest is weaker (B4).
(`oob_score_` is the OOB *accuracy*, 0.893: barely above the 0.88 of "nobody churns", and not useful to judge a churn model — Lab 4.)
To compare the forest with the other models we keep CV: same folds, same measure, paired differences.

## B3. Ablation: the two sources of randomness

A random forest has two sources of diversity among its trees (Lecture 13): **bootstrap** samples of the records, and a random
**subset of attributes** at each split (`max_features`). Remove them one at a time, and both together:

| variant | change w.r.t. `rf_13` |
|---|---|
| random forest (baseline) | — |
| no bootstrap | `bootstrap=False`: every tree sees all 2100 customers |
| no attribute subsets | `max_features=None`: every split sees all 8 attributes (= bagging of trees with leaf 5) |
| neither | `bootstrap=False, max_features=None` |
| one tree, leaf 5 | `pipe(DecisionTreeClassifier(min_samples_leaf=5, random_state=0))` |

Keep `n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1` everywhere.

In [7]:
def rf(**kw):
    return pipe(RandomForestClassifier(**{"n_estimators": 300, "max_features": 1, "min_samples_leaf": 5,
                                          "random_state": 0, "n_jobs": -1, **kw}))

ablation({"random forest": (rf(), Xp_train),
          "no bootstrap": (rf(bootstrap=False), Xp_train),
          "no attribute subsets": (rf(max_features=None), Xp_train),
          "neither": (rf(bootstrap=False, max_features=None), Xp_train),
          "one tree, leaf 5": (pipe(DecisionTreeClassifier(min_samples_leaf=5, random_state=0)), Xp_train)})

,CV AUC,std,diff,std diff,folds worse
random forest,0.812,0.028,0.000,0.000,0
no bootstrap,0.803,0.026,-0.009,0.004,5
no attribute subsets,0.792,0.031,-0.020,0.011,5
neither,0.704,0.045,-0.108,0.032,5
"one tree, leaf 5",0.696,0.045,-0.116,0.035,5


**Questions.**

1. Which source of randomness contributes more on NordicMart? Is the effect consistent across the folds?
1. Without either source, the 300 trees are (almost) identical. Check it against the last row. What is left of the forest?
1. Is the effect of removing both the sum of the two single effects? Explain with the conditions of Lecture 13, *Two conditions*.

**Answers.**

1. The attribute subsets: removing them costs 0.020 (worse in 5 folds of 5), removing the bootstrap costs 0.009 (5 of 5, with a very small `std diff`).
   Recency dominates the first splits (Lecture 13): without `max_features` every tree starts in the same way, and the trees make the same errors.
1. "Neither" gives 0.704, close to the single tree with leaf 5 (0.696): 300 copies of the same tree vote like one tree. (They are not exactly identical:
   attributes are visited in a random order and ties between equally good splits are broken differently.) A full tree with tiny leaves is the
   unstable, overfitting model of Lecture 09.
1. No: 0.009 + 0.020 ≈ 0.03, but removing both costs 0.108. Each source alone is enough to make the trees **diverse**, and diversity is what turns
   a vote of overfitting trees into a good model. Only when both are removed does diversity vanish. A typical interaction: the 2 × 2 design shows it,
   two single ablations from the baseline would have hidden it.